# Phase 7 — Variation du learning rate

## Objectif

Mesurer l’impact du learning rate sur la convergence.

On compare trois valeurs :
- `1e-7` : trop petit
- `1e-3` : sweet spot
- `1.0` : trop grand

On garde :
- les mêmes données ;
- la même architecture ;
- le même optimizer (Adam) ;
- seule la valeur du learning rate change.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import time

In [2]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test = X_test.reshape(-1, 784).astype("float32") / 255.0

print(f"Train : {X_train.shape} | Test : {X_test.shape}")
print(f"Classes uniques : {np.unique(y_train)}")

Train : (60000, 784) | Test : (10000, 784)
Classes uniques : [0 1 2 3 4 5 6 7 8 9]


## 1. Réglages de l’expérience

Learning rates testés :
- `1e-7`
- `1e-3`
- `1.0`

Labels :
- trop petit
- sweet spot
- trop grand

In [3]:
learning_rates = [1e-7, 1e-3, 1.0]
lr_labels = ["trop petit (1e-7)", "sweet spot (1e-3)", "trop grand (1.0)"]

results = []
histories = {}

In [4]:
for lr, label in zip(learning_rates, lr_labels):
    tf.random.set_seed(42)

    model = keras.Sequential([
        keras.layers.Dense(128, activation="relu", input_shape=(784,)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    start = time.time()

    history = model.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=64,
        validation_split=0.1,
        verbose=0
    )

    train_time = time.time() - start
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    val_loss_final = history.history["val_loss"][-1]

    results.append({
        "lr": lr,
        "label": label,
        "val_loss_final": val_loss_final,
        "test_accuracy": test_acc,
        "train_time_s": train_time
    })

    histories[label] = history.history["val_loss"]

d:\Cours\5ème Année IPSSI - Paris\Machine Learning\deeplearning-j1-pmc-from-scratch-DONOU-Serge\.venv311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
print("\n=== TABLEAU COMPARATIF LEARNING RATE ===")
print(f"{'LR':8s} | {'Label':24s} | {'Val loss final':14s} | {'Test acc':10s} | {'Temps (s)':10s}")
print("-" * 80)

for r in results:
    print(
        f"{r['lr']:.0e} | "
        f"{r['label']:24s} | "
        f"{r['val_loss_final']:.4f} | "
        f"{r['test_accuracy']:.4f} | "
        f"{r['train_time_s']:.0f}"
    )


=== TABLEAU COMPARATIF LEARNING RATE ===
LR       | Label                    | Val loss final | Test acc   | Temps (s) 
--------------------------------------------------------------------------------
1e-07 | trop petit (1e-7)        | 2.2755 | 0.1165 | 38
1e-03 | sweet spot (1e-3)        | 0.1095 | 0.9766 | 33
1e+00 | trop grand (1.0)         | 2.3733 | 0.0980 | 35


In [6]:
plt.figure(figsize=(10, 5))

for label, val_losses in histories.items():
    plt.plot(range(1, 11), val_losses, label=label, linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Val Loss")
plt.title("Impact du learning rate sur la convergence (MNIST)")
plt.legend()
plt.yscale("log")

plt.savefig("phase7_lr_curve.png", dpi=100, bbox_inches="tight")
plt.show()

print("\nCourbe sauvegardée : phase7_lr_curve.png")


Courbe sauvegardée : phase7_lr_curve.png


C:\Users\serge\AppData\Local\Temp\ipykernel_11956\545643087.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Analyse attendue

### Scénario normal
- `lr = 1e-3` doit converger proprement ;
- `lr = 1e-7` apprend presque rien en 10 epochs ;
- `lr = 1.0` oscille ou diverge.

### Interprétation
- un learning rate trop petit produit des mises à jour presque nulles ;
- un learning rate bien choisi descend vite et proprement ;
- un learning rate trop grand saute au-dessus des minima.

### Conclusion
Le learning rate est l’un des hyperparamètres les plus importants pour la stabilité et la vitesse d’apprentissage.